<a href="https://colab.research.google.com/github/luciagorostidi/solar_panels_defects/blob/main/TFM_YOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Thu Aug 27 08:47:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4


In [ ]:
!pip install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 3.4 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import ultralytics

print("Versión de Ultralytics:", ultralytics.__version__)
print("YOLO cargado correctamente.")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Versión de Ultralytics: 8.4.129
YOLO cargado correctamente.


In [ ]:
!pip install -q roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.4 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import roboflow

baseline_version = 2

#para recuperar la API Key del baseline en roboflow que se ha guardado en Colab
api_key = userdata.get("ROBOFLOW_API_KEY")

#conectar con roboflow
rf = roboflow.Roboflow(api_key=api_key)

#acceder al proyecto
project = rf.workspace ("luca-gorostidi-garca-s-workspace").project("solar-panels-cyg5d")

#seleccionar la versión del dataset "baseline"
version = project.version(baseline_version)

dataset = version.download("yolov11")

print("Dataset descargado correctamente")
print("Ubicación:", dataset.location)


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Solar-panels-2 in yolov11:: 100%|██████████| 1453/1453 [00:00<00:00, 5660.70it/s]

Dataset descargado correctamente
Ubicación: /content/Solar-panels-2


In [ ]:
print("Data set descargado correctamente")
print("Ubicación:", dataset.location)

Data set descargado correctamente
Ubicación: /content/Solar-panels-2


In [ ]:
import os
import yaml

dataset_path = dataset.location
yaml_path = os.path.join(dataset_path, "data.yaml")

#leer archivo data.yaml
with open(yaml_path, "r") as f:
  data_config = yaml.safe_load(f)

print("=== CONTENIDO DE data.yaml ===")
print(data_config)

#ver cuántas imágenes hay en train,valid y test
for split in ["train","valid","test"]:
  images_path = os.path.join(dataset_path,split,"images")

  if os.path.exists(images_path):
      n_images = len([
          f for f in os.listdir(images_path)
          if f.lower().endswith((".jpg",".jpeg",".png",".webp"))
      ])
      print(f"{split}: {n_images} imágenes")
  else:
      print(f"{split}: carpeta no encontrada")



=== CONTENIDO DE data.yaml ===
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 6, 'names': ['brid_droppings', 'crack', 'dust', 'electro', 'snow', 'thermal_anomaly'], 'roboflow': {'workspace': 'luca-gorostidi-garca-s-workspace', 'project': 'solar-panels-cyg5d', 'version': 2, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/luca-gorostidi-garca-s-workspace/solar-panels-cyg5d/dataset/2'}}
train: 507 imágenes
valid: 145 imágenes
test: 72 imágenes


In [ ]:
print("Número de clases:", data_config["nc"])
print("\nClases del dataset:")

for i, clase in enumerate(data_config["names"]):
  print(f"{i}: {clase}")

Número de clases: 6

Clases del dataset:
0: brid_droppings
1: crack
2: dust
3: electro
4: snow
5: thermal_anomaly


In [ ]:
!pip show ultralytics

Name: ultralytics
Version: 8.4.129
Summary: Ultralytics YOLO 🚀 for SOTA object detection, multi-object tracking, instance segmentation, pose estimation, classification, and oriented object detection.
Home-page: https://ultralytics.com
Author: 
Author-email: Glenn Jocher <glenn.jocher@ultralytics.com>, Jing Qiu <jing.qiu@ultralytics.com>
License: AGPL-3.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: filelock, matplotlib, numpy, nvidia-ml-py, opencv-python, pillow, polars, psutil, pyyaml, requests, torch, torchvision, ultralytics-platform, ultralytics-thop
Required-by: 


In [ ]:
!pip install -q -U ultralytics

In [ ]:
import ultralytics
print("Versión:", ultralytics.__version__)

Versión: 8.4.129


In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")

print("Modelo YOLO11n cargado correctamente.")

Modelo YOLO11n cargado correctamente.


In [ ]:
!pip show ultralytics

Name: ultralytics
Version: 8.4.129
Summary: Ultralytics YOLO 🚀 for SOTA object detection, multi-object tracking, instance segmentation, pose estimation, classification, and oriented object detection.
Home-page: https://ultralytics.com
Author: 
Author-email: Glenn Jocher <glenn.jocher@ultralytics.com>, Jing Qiu <jing.qiu@ultralytics.com>
License: AGPL-3.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: filelock, matplotlib, numpy, nvidia-ml-py, opencv-python, pillow, polars, psutil, pyyaml, requests, torch, torchvision, ultralytics-platform, ultralytics-thop
Required-by: 


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

ruta_experimentos = "/content/drive/MyDrive/TFM/Experimentos"

print("Carpeta encontrada:", os.path.exists(ruta_experimentos))

Carpeta encontrada: True


In [ ]:
results = model.train(
    data=yaml_path,
    epochs=100,
    patience=20,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/drive/MyDrive/TFM/Experimentos",
    name="E1_baseline_100ep_pat20_yolo11n",
    seed=42,
    deterministic=True,
    plots=True
)

Ultralytics 8.4.129 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Solar-panels-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=E1_baseline_100ep_pat20_yo

In [ ]:
import pandas as pd

ruta_csv = "/content/drive/MyDrive/TFM/Experimentos/E1_baseline_100ep_pat20_yolo11n/results.csv"

df = pd.read_csv(ruta_csv)

print(df.head())
print(df.columns)

   epoch     time  train/box_loss  train/cls_loss  train/dfl_loss  \
0      1  39.0784         1.14434         3.82704         1.60420   
1      2  48.6871         1.18216         3.43633         1.59203   
2      3  59.1184         1.28760         3.12123         1.67081   
3      4  67.5192         1.25607         2.93109         1.66193   
4      5  78.4964         1.23056         2.89854         1.66146   

   metrics/precision(B)  metrics/recall(B)  metrics/mAP50(B)  \
0               0.00333            0.52339           0.06878   
1               0.00167            0.58049           0.04689   
2               0.38007            0.07997           0.07970   
3               0.40891            0.19812           0.10103   
4               0.30563            0.26864           0.11497   

   metrics/mAP50-95(B)  val/box_loss  val/cls_loss  val/dfl_loss    lr/pg0  \
0              0.04056       1.37518       7.06844       1.80421  0.000323   
1              0.01244       2.06196       7

In [ ]:
# Mejor época según mAP50-95
idx_best = df["metrics/mAP50-95(B)"].idxmax()
best = df.loc[idx_best]

print("Épocas completadas:", len(df))
print("Mejor época:", int(best["epoch"]))
print("Precision:", round(best["metrics/precision(B)"], 4))
print("Recall:", round(best["metrics/recall(B)"], 4))
print("mAP50:", round(best["metrics/mAP50(B)"], 4))
print("mAP50-95:", round(best["metrics/mAP50-95(B)"], 4))

print("\nMáximo mAP50 alcanzado:",
      round(df["metrics/mAP50(B)"].max(), 4),
      "en la época",
      int(df.loc[df["metrics/mAP50(B)"].idxmax(), "epoch"]))

Épocas completadas: 81
Mejor época: 61
Precision: 0.3728
Recall: 0.4514
mAP50: 0.3958
mAP50-95: 0.2363

Máximo mAP50 alcanzado: 0.3958 en la época 61


In [ ]:
last = df.iloc[-1]

print("Última época:", int(last["epoch"]))
print("Precision:", round(last["metrics/precision(B)"], 4))
print("Recall:", round(last["metrics/recall(B)"], 4))
print("mAP50:", round(last["metrics/mAP50(B)"], 4))
print("mAP50-95:", round(last["metrics/mAP50-95(B)"], 4))

Última época: 81
Precision: 0.35
Recall: 0.4105
mAP50: 0.3346
mAP50-95: 0.2258


In [ ]:
from ultralytics import YOLO

best_model = YOLO(
    "/content/drive/MyDrive/TFM/Experimentos/E1_baseline_100ep_pat20_yolo11n/weights/best.pt"
)

test_results = best_model.val(
    data=yaml_path,
    split="test",
    device=0
)

Ultralytics 8.4.129 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1709.6±490.5 MB/s, size: 73.5 KB)
val: Scanning /content/Solar-panels-2/test/labels... 72 images, 22 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 72/72 2.4Kit/s 0.0s
val: New cache created: /content/Solar-panels-2/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.3it/s 2.2s
                   all         72         65      0.366      0.457      0.429      0.315
        brid_droppings         13         15      0.376        0.6      0.417      0.334
                 crack          5          6      0.198      0.167      0.191      0.174
                  dust         11         11       0.38      0.636      0.669       0.56
               electro          6          9      0.113      0.111      0

In [ ]:
test_results = best_model.val(
    data=yaml_path,
    split="test",
    device=0,
    plots=True,
    project="/content/drive/MyDrive/TFM/Experimentos/E1_baseline_100ep_pat20_yolo11n",
    name="test"
)

Ultralytics 8.4.129 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1860.9±564.8 MB/s, size: 62.5 KB)
val: Scanning /content/Solar-panels-2/test/labels.cache... 72 images, 22 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 72/72 20.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.1it/s 2.4s
                   all         72         65      0.366      0.457      0.429      0.315
        brid_droppings         13         15      0.376        0.6      0.417      0.334
                 crack          5          6      0.198      0.167      0.191      0.174
                  dust         11         11       0.38      0.636      0.669       0.56
               electro          6          9      0.113      0.111      0.036    0.00839
                  snow         12         12      0.728      0.894      0.875       0.68
       thermal_anomaly          6    

Una vez terminado el entrenamiento, cambio el entorno de ejecución a CPU para no gastar más sesión en GPU.

El problema es que, tras cambiarlo, me voy cuenta de que no he registrado el tiempo de entrenamiento. No pasa nada, con la propia CPU vuelvo a conectarme con drive y lo saco:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

ruta_csv = "/content/drive/MyDrive/TFM/Experimentos/E1_baseline_100ep_pat20_yolo11n/results.csv"
df = pd.read_csv(ruta_csv)

tiempo_segundos = df["time"].iloc[-1]

horas = int(tiempo_segundos // 3600)
minutos = int((tiempo_segundos % 3600) // 60)
segundos = tiempo_segundos % 60

print(f"Tiempo total: {horas} h {minutos} min {segundos:.1f} s")

Tiempo total: 0 h 14 min 21.9 s


Empieza el experimento 2. Se mantiene la misma configuración experimental utilizada en E1, solamente se modifica la versión del dataset empleada para el entrenamiento. Esto quiere decir que se pueden reutilizar muchas celdas de la sesión, no se copian de nuevo, simplemente se vuelven a ejecutar.

Lo único que hay que modificar es la versión del dataset.

E1 → version(2) → baseline_clean
E2 → version(3) → E2_expanded_dataset


In [ ]:
from google.colab import userdata
import roboflow

e2_version = 3

# recuperar la misma API Key guardada en Colab
api_key = userdata.get("ROBOFLOW_API_KEY")

# conectar con Roboflow
rf = roboflow.Roboflow(api_key=api_key)

# acceder al mismo proyecto
project = rf.workspace("luca-gorostidi-garca-s-workspace").project("solar-panels-cyg5d")

# seleccionar la versión E2
version = project.version(e2_version)

dataset = version.download("yolov11")

print("Dataset E2 descargado correctamente")
print("Ubicación:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Solar-panels-3 in yolov11:: 100%|██████████| 1783/1783 [00:00<00:00, 5829.07it/s]

Dataset E2 descargado correctamente
Ubicación: /content/Solar-panels-3


In [ ]:
print("Data set descargado correctamente")
print("Ubicación:", dataset.location)

Data set descargado correctamente
Ubicación: /content/Solar-panels-3


In [ ]:
import os
import yaml

dataset_path = dataset.location
yaml_path = os.path.join(dataset_path, "data.yaml")

#leer archivo data.yaml
with open(yaml_path, "r") as f:
  data_config = yaml.safe_load(f)

print("=== CONTENIDO DE data.yaml ===")
print(data_config)

#ver cuántas imágenes hay en train,valid y test
for split in ["train","valid","test"]:
  images_path = os.path.join(dataset_path,split,"images")

  if os.path.exists(images_path):
      n_images = len([
          f for f in os.listdir(images_path)
          if f.lower().endswith((".jpg",".jpeg",".png",".webp"))
      ])
      print(f"{split}: {n_images} imágenes")
  else:
      print(f"{split}: carpeta no encontrada")

=== CONTENIDO DE data.yaml ===
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 6, 'names': ['brid_droppings', 'crack', 'dust', 'electro', 'snow', 'thermal_anomaly'], 'roboflow': {'workspace': 'luca-gorostidi-garca-s-workspace', 'project': 'solar-panels-cyg5d', 'version': 3, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/luca-gorostidi-garca-s-workspace/solar-panels-cyg5d/dataset/3'}}
train: 672 imágenes
valid: 145 imágenes
test: 72 imágenes


In [ ]:
!pip show ultralytics

Name: ultralytics
Version: 8.4.129
Summary: Ultralytics YOLO 🚀 for SOTA object detection, multi-object tracking, instance segmentation, pose estimation, classification, and oriented object detection.
Home-page: https://ultralytics.com
Author: 
Author-email: Glenn Jocher <glenn.jocher@ultralytics.com>, Jing Qiu <jing.qiu@ultralytics.com>
License: AGPL-3.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: filelock, matplotlib, numpy, nvidia-ml-py, opencv-python, pillow, polars, psutil, pyyaml, requests, torch, torchvision, ultralytics-platform, ultralytics-thop
Required-by: 


In [ ]:
!pip install -q -U ultralytics

In [ ]:
import ultralytics
print("Versión:", ultralytics.__version__)

Versión: 8.4.129


Para E2 se carga otro YOLO11N preentrenado desde cero respecto a E1.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")

print("Modelo YOLO11n cargado correctamente.")

Modelo YOLO11n cargado correctamente.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

ruta_experimentos = "/content/drive/MyDrive/TFM/Experimentos"

print("Carpeta encontrada:", os.path.exists(ruta_experimentos))

Carpeta encontrada: True


In [ ]:
results = model.train(
    data=yaml_path,
    epochs=100,
    patience=20,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/drive/MyDrive/TFM/Experimentos",
    name="E2_extended_100ep_pat20_yolo11n",
    seed=42,
    deterministic=True,
    plots=True
)

Ultralytics 8.4.129 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Solar-panels-3/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=E2_extended_100ep_pat20_yo

In [ ]:
import os

ruta_experimentos = "/content/drive/MyDrive/TFM/Experimentos"

for carpeta in os.listdir(ruta_experimentos):
    print(carpeta)

E1_baseline_100ep_pat20_yolo11n
E2_extended_100ep_pat20_yolo11n


In [ ]:
import pandas as pd

ruta_csv = "/content/drive/MyDrive/TFM/Experimentos/E2_extended_100ep_pat20_yolo11n/results.csv"

df_e2 = pd.read_csv(ruta_csv)

idx_best = df_e2["metrics/mAP50-95(B)"].idxmax()
best_e2 = df_e2.loc[idx_best]
last_e2 = df_e2.iloc[-1]

print("Épocas completadas:", len(df_e2))

print("\n--- MEJOR ÉPOCA ---")
print("Época:", int(best_e2["epoch"]))
print("Precision:", round(best_e2["metrics/precision(B)"], 4))
print("Recall:", round(best_e2["metrics/recall(B)"], 4))
print("mAP50:", round(best_e2["metrics/mAP50(B)"], 4))
print("mAP50-95:", round(best_e2["metrics/mAP50-95(B)"], 4))

print("\nMáximo mAP50:",
      round(df_e2["metrics/mAP50(B)"].max(), 4),
      "en época",
      int(df_e2.loc[df_e2["metrics/mAP50(B)"].idxmax(), "epoch"]))

print("\n--- ÚLTIMA ÉPOCA ---")
print("Época:", int(last_e2["epoch"]))
print("Precision:", round(last_e2["metrics/precision(B)"], 4))
print("Recall:", round(last_e2["metrics/recall(B)"], 4))
print("mAP50:", round(last_e2["metrics/mAP50(B)"], 4))
print("mAP50-95:", round(last_e2["metrics/mAP50-95(B)"], 4))

tiempo = df_e2["time"].iloc[-1]
print("\nTiempo total:",
      int(tiempo // 60), "min",
      round(tiempo % 60, 1), "s")

Épocas completadas: 99

--- MEJOR ÉPOCA ---
Época: 79
Precision: 0.3755
Recall: 0.4736
mAP50: 0.4147
mAP50-95: 0.2802

Máximo mAP50: 0.4147 en época 79

--- ÚLTIMA ÉPOCA ---
Época: 99
Precision: 0.3524
Recall: 0.4363
mAP50: 0.3515
mAP50-95: 0.2372

Tiempo total: 22 min 21.5 s


In [ ]:
from ultralytics import YOLO

best_model_e2 = YOLO(
    "/content/drive/MyDrive/TFM/Experimentos/E2_extended_100ep_pat20_yolo11n/weights/best.pt"
)

test_results_e2 = best_model_e2.val(
    data=yaml_path,
    split="test",
    device=0,
    plots=True,
    project="/content/drive/MyDrive/TFM/Experimentos/E2_extended_100ep_pat20_yolo11n",
    name="test"
)

Ultralytics 8.4.129 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1466.1±405.1 MB/s, size: 63.0 KB)
val: Scanning /content/Solar-panels-3/test/labels... 72 images, 22 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 72/72 2.1Kit/s 0.0s
val: New cache created: /content/Solar-panels-3/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.3it/s 3.7s
                   all         72         65       0.39      0.481      0.431      0.319
        brid_droppings         13         15      0.373      0.467      0.456      0.358
                 crack          5          6      0.272      0.167      0.138      0.103
                  dust         11         11       0.43      0.727       0.64      0.551
               electro          6          9      0.154      0.111      0

Experimento E3 - Revissed Classes

En este experimento se utiliza la versión E3_revised_classes, obtenida a partir de la revisión de los resultados de E1 y E2. El dataset queda reducido a tres clases: soiling, crack y snow, tras fusionar dust y bird_droppings y eliminar electro y thermal_anomaly.

In [ ]:
from google.colab import userdata
import roboflow

e3_version = 4

api_key = userdata.get("ROBOFLOW_API_KEY")

rf = roboflow.Roboflow(api_key=api_key)

project = rf.workspace(
    "luca-gorostidi-garca-s-workspace"
).project("solar-panels-cyg5d")

version = project.version(e3_version)

dataset = version.download("yolov11")

print("Dataset E3 descargado correctamente")
print("Ubicación:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov11 in progress : 0.1%
Version export complete for yolov11 format



Extracting Dataset Version Zip to Solar-panels-4 in yolov11:: 100%|██████████| 1519/1519 [00:00<00:00, 5638.83it/s]

Dataset E3 descargado correctamente
Ubicación: /content/Solar-panels-4


In [ ]:
print("Data set descargado correctamente")
print("Ubicación:", dataset.location)

Data set descargado correctamente
Ubicación: /content/Solar-panels-4


In [ ]:
import os
import yaml

dataset_path = dataset.location
yaml_path = os.path.join(dataset_path, "data.yaml")

#leer archivo data.yaml
with open(yaml_path, "r") as f:
  data_config = yaml.safe_load(f)

print("=== CONTENIDO DE data.yaml ===")
print(data_config)

#ver cuántas imágenes hay en train,valid y test
for split in ["train","valid","test"]:
  images_path = os.path.join(dataset_path,split,"images")

  if os.path.exists(images_path):
      n_images = len([
          f for f in os.listdir(images_path)
          if f.lower().endswith((".jpg",".jpeg",".png",".webp"))
      ])
      print(f"{split}: {n_images} imágenes")
  else:
      print(f"{split}: carpeta no encontrada")

=== CONTENIDO DE data.yaml ===
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 3, 'names': ['crack', 'snow', 'soiling'], 'roboflow': {'workspace': 'luca-gorostidi-garca-s-workspace', 'project': 'solar-panels-cyg5d', 'version': 4, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/luca-gorostidi-garca-s-workspace/solar-panels-cyg5d/dataset/4'}}
train: 573 imágenes
valid: 122 imágenes
test: 62 imágenes


In [ ]:
!pip show ultralytics

Name: ultralytics
Version: 8.4.129
Summary: Ultralytics YOLO 🚀 for SOTA object detection, multi-object tracking, instance segmentation, pose estimation, classification, and oriented object detection.
Home-page: https://ultralytics.com
Author: 
Author-email: Glenn Jocher <glenn.jocher@ultralytics.com>, Jing Qiu <jing.qiu@ultralytics.com>
License: AGPL-3.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: filelock, matplotlib, numpy, nvidia-ml-py, opencv-python, pillow, polars, psutil, pyyaml, requests, torch, torchvision, ultralytics-platform, ultralytics-thop
Required-by: 


In [ ]:
!pip install -q -U ultralytics

In [ ]:
import ultralytics
print("Versión:", ultralytics.__version__)

Versión: 8.4.129


Se carga otra versión de YOLO11n preentrenado

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")

print("Modelo YOLO11n cargado correctamente.")

Modelo YOLO11n cargado correctamente.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

ruta_experimentos = "/content/drive/MyDrive/TFM/Experimentos"

print("Carpeta encontrada:", os.path.exists(ruta_experimentos))

Carpeta encontrada: True


In [ ]:
results = model.train(
    data=yaml_path,
    epochs=100,
    patience=20,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/drive/MyDrive/TFM/Experimentos",
    name="E3_revised_100ep_pat20_yolo11n",
    seed=42,
    deterministic=True,
    plots=True
)

Ultralytics 8.4.129 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Solar-panels-4/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=E3_revised_100ep_pat20_yol

In [ ]:
import os

ruta_experimentos = "/content/drive/MyDrive/TFM/Experimentos"

for carpeta in os.listdir(ruta_experimentos):
    print(carpeta)

E1_baseline_100ep_pat20_yolo11n
E2_extended_100ep_pat20_yolo11n
E3_revised_100ep_pat20_yolo11n


In [ ]:
import pandas as pd

ruta_csv = "/content/drive/MyDrive/TFM/Experimentos/E3_revised_100ep_pat20_yolo11n/results.csv"

df_e2 = pd.read_csv(ruta_csv)

idx_best = df_e2["metrics/mAP50-95(B)"].idxmax()
best_e2 = df_e2.loc[idx_best]
last_e2 = df_e2.iloc[-1]

print("Épocas completadas:", len(df_e2))

print("\n--- MEJOR ÉPOCA ---")
print("Época:", int(best_e2["epoch"]))
print("Precision:", round(best_e2["metrics/precision(B)"], 4))
print("Recall:", round(best_e2["metrics/recall(B)"], 4))
print("mAP50:", round(best_e2["metrics/mAP50(B)"], 4))
print("mAP50-95:", round(best_e2["metrics/mAP50-95(B)"], 4))

print("\nMáximo mAP50:",
      round(df_e2["metrics/mAP50(B)"].max(), 4),
      "en época",
      int(df_e2.loc[df_e2["metrics/mAP50(B)"].idxmax(), "epoch"]))

print("\n--- ÚLTIMA ÉPOCA ---")
print("Época:", int(last_e2["epoch"]))
print("Precision:", round(last_e2["metrics/precision(B)"], 4))
print("Recall:", round(last_e2["metrics/recall(B)"], 4))
print("mAP50:", round(last_e2["metrics/mAP50(B)"], 4))
print("mAP50-95:", round(last_e2["metrics/mAP50-95(B)"], 4))

tiempo = df_e2["time"].iloc[-1]
print("\nTiempo total:",
      int(tiempo // 60), "min",
      round(tiempo % 60, 1), "s")

Épocas completadas: 81

--- MEJOR ÉPOCA ---
Época: 61
Precision: 0.5148
Recall: 0.4298
mAP50: 0.4272
mAP50-95: 0.3137

Máximo mAP50: 0.4426 en época 69

--- ÚLTIMA ÉPOCA ---
Época: 81
Precision: 0.5032
Recall: 0.39
mAP50: 0.409
mAP50-95: 0.2774

Tiempo total: 15 min 31.4 s


In [ ]:
from ultralytics import YOLO

best_model_e2 = YOLO(
    "/content/drive/MyDrive/TFM/Experimentos/E3_revised_100ep_pat20_yolo11n/weights/best.pt"
)

test_results_e2 = best_model_e2.val(
    data=yaml_path,
    split="test",
    device=0,
    plots=True,
    project="/content/drive/MyDrive/TFM/Experimentos/E3_revised_100ep_pat20_yolo11n",
    name="test"
)

Ultralytics 8.4.129 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1513.3±272.9 MB/s, size: 88.6 KB)
val: Scanning /content/Solar-panels-4/test/labels... 62 images, 22 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 62/62 2.2Kit/s 0.0s
val: New cache created: /content/Solar-panels-4/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.2it/s 3.3s
                   all         62         45      0.693      0.466      0.585      0.449
                 crack          5          7      0.411      0.143      0.244      0.165
                  snow         12         12      0.882      0.833      0.924      0.701
               soiling         23         26      0.786      0.423      0.586       0.48
Speed: 11.5ms preprocess, 11.2ms inference, 0.0ms loss, 5.1ms postprocess

Experimento E4 – Augmented Dataset

E4 parte de la misma estructura de clases definida en E3 y conserva sin cambios los conjuntos de validation y test. La diferencia se introduce exclusivamente en el conjunto de entrenamiento, que se amplía mediante técnicas de data augmentation aplicadas en Roboflow.

In [ ]:
from google.colab import userdata
import roboflow

e4_version = 5

api_key = userdata.get("ROBOFLOW_API_KEY")

rf = roboflow.Roboflow(api_key=api_key)

project = rf.workspace(
    "luca-gorostidi-garca-s-workspace"
).project("solar-panels-cyg5d")

version = project.version(e4_version)

dataset = version.download("yolov11")

print("Dataset E4 descargado correctamente")
print("Ubicación:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov11 in progress : 60.0%
Version export complete for yolov11 format



Extracting Dataset Version Zip to Solar-panels-5 in yolov11:: 100%|██████████| 3811/3811 [00:00<00:00, 5528.69it/s]


Dataset E4 descargado correctamente
Ubicación: /content/Solar-panels-5


In [ ]:
import os
import yaml

dataset_path = dataset.location
yaml_path = os.path.join(dataset_path, "data.yaml")

#leer archivo data.yaml
with open(yaml_path, "r") as f:
  data_config = yaml.safe_load(f)

print("=== CONTENIDO DE data.yaml ===")
print(data_config)

#ver cuántas imágenes hay en train,valid y test
for split in ["train","valid","test"]:
  images_path = os.path.join(dataset_path,split,"images")

  if os.path.exists(images_path):
      n_images = len([
          f for f in os.listdir(images_path)
          if f.lower().endswith((".jpg",".jpeg",".png",".webp"))
      ])
      print(f"{split}: {n_images} imágenes")
  else:
      print(f"{split}: carpeta no encontrada")

=== CONTENIDO DE data.yaml ===
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 3, 'names': ['crack', 'snow', 'soiling'], 'roboflow': {'workspace': 'luca-gorostidi-garca-s-workspace', 'project': 'solar-panels-cyg5d', 'version': 5, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/luca-gorostidi-garca-s-workspace/solar-panels-cyg5d/dataset/5'}}
train: 1719 imágenes
valid: 122 imágenes
test: 62 imágenes


In [ ]:
!pip show ultralytics

Name: ultralytics
Version: 8.4.130
Summary: Ultralytics YOLO 🚀 for SOTA object detection, multi-object tracking, instance segmentation, pose estimation, classification, and oriented object detection.
Home-page: https://ultralytics.com
Author: 
Author-email: Glenn Jocher <glenn.jocher@ultralytics.com>, Jing Qiu <jing.qiu@ultralytics.com>
License: AGPL-3.0
Location: /usr/local/lib/python3.13/dist-packages
Requires: filelock, matplotlib, numpy, nvidia-ml-py, opencv-python, pillow, polars, psutil, pyyaml, requests, torch, torchvision, ultralytics-platform, ultralytics-thop
Required-by: 


In [ ]:
!pip install -q -U ultralytics

In [ ]:
import ultralytics
print("Versión:", ultralytics.__version__)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Versión: 8.4.130


Se carga otra versi´on de YOLO11n preentrenadp

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")

print("Modelo YOLO11n cargado correctamente.")

Modelo YOLO11n cargado correctamente.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

ruta_experimentos = "/content/drive/MyDrive/TFM/Experimentos"

print("Carpeta encontrada:", os.path.exists(ruta_experimentos))

Carpeta encontrada: True


In [ ]:
results = model.train(
    data=yaml_path,
    epochs=100,
    patience=20,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/drive/MyDrive/TFM/Experimentos",
    name="E4_augmentation_100ep_pat20_yolo11n",
    seed=42,
    deterministic=True,
    plots=True
)

Ultralytics 8.4.130 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Solar-panels-5/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=E4_augmentation_100ep_pat2

In [ ]:
import os

ruta_experimentos = "/content/drive/MyDrive/TFM/Experimentos"

for carpeta in os.listdir(ruta_experimentos):
    print(carpeta)

E1_baseline_100ep_pat20_yolo11n
E2_extended_100ep_pat20_yolo11n
E3_revised_100ep_pat20_yolo11n
E4_augmentation_100ep_pat20_yolo11n


In [ ]:
import pandas as pd

ruta_csv = "/content/drive/MyDrive/TFM/Experimentos/E4_augmentation_100ep_pat20_yolo11n/results.csv"

df_e2 = pd.read_csv(ruta_csv)

idx_best = df_e2["metrics/mAP50-95(B)"].idxmax()
best_e2 = df_e2.loc[idx_best]
last_e2 = df_e2.iloc[-1]

print("Épocas completadas:", len(df_e2))

print("\n--- MEJOR ÉPOCA ---")
print("Época:", int(best_e2["epoch"]))
print("Precision:", round(best_e2["metrics/precision(B)"], 4))
print("Recall:", round(best_e2["metrics/recall(B)"], 4))
print("mAP50:", round(best_e2["metrics/mAP50(B)"], 4))
print("mAP50-95:", round(best_e2["metrics/mAP50-95(B)"], 4))

print("\nMáximo mAP50:",
      round(df_e2["metrics/mAP50(B)"].max(), 4),
      "en época",
      int(df_e2.loc[df_e2["metrics/mAP50(B)"].idxmax(), "epoch"]))

print("\n--- ÚLTIMA ÉPOCA ---")
print("Época:", int(last_e2["epoch"]))
print("Precision:", round(last_e2["metrics/precision(B)"], 4))
print("Recall:", round(last_e2["metrics/recall(B)"], 4))
print("mAP50:", round(last_e2["metrics/mAP50(B)"], 4))
print("mAP50-95:", round(last_e2["metrics/mAP50-95(B)"], 4))

tiempo = df_e2["time"].iloc[-1]
print("\nTiempo total:",
      int(tiempo // 60), "min",
      round(tiempo % 60, 1), "s")

Épocas completadas: 100

--- MEJOR ÉPOCA ---
Época: 83
Precision: 0.5857
Recall: 0.6011
mAP50: 0.5374
mAP50-95: 0.3601

Máximo mAP50: 0.5374 en época 83

--- ÚLTIMA ÉPOCA ---
Época: 100
Precision: 0.5509
Recall: 0.5297
mAP50: 0.4594
mAP50-95: 0.3356

Tiempo total: 57 min 24.4 s


In [ ]:
from ultralytics import YOLO

best_model_e2 = YOLO(
    "/content/drive/MyDrive/TFM/Experimentos/E4_augmentation_100ep_pat20_yolo11n/weights/best.pt"
)

test_results_e2 = best_model_e2.val(
    data=yaml_path,
    split="test",
    device=0,
    plots=True,
    project="/content/drive/MyDrive/TFM/Experimentos/E4_augmentation_100ep_pat20_yolo11n",
    name="test"
)

Ultralytics 8.4.130 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.4 GFLOPs
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1557.6±258.0 MB/s, size: 96.2 KB)
test: Scanning /content/Solar-panels-5/test/labels.cache... 62 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 62/62 11.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.0it/s 3.9s
                   all         62         44      0.615      0.631      0.558      0.417
                 crack          5          7       0.42      0.429      0.192      0.141
                  snow         12         12      0.794      0.917      0.912      0.627
               soiling         22         25      0.632      0.549      0.571      0.485
Speed: 13.6ms preprocess, 7.5ms inference, 0.0ms loss, 8.0ms postprocess per image
Results saved to /content/drive/MyDrive/TFM/Exp

Descargo el archivo en .onnx

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics onnx onnxruntime

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 7.7 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO


weights_path = '/content/drive/MyDrive/TFM/Experimentos/E4_augmentation_100ep_pat20_yolo11n/weights/best.pt'

# Cargar y exportar a 640x640
model = YOLO(weights_path)
model.export(format='onnx', imgsz=640)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.132 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/TFM/Experimentos/E4_augmentation_100ep_pat20_yolo11n/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (5.2 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 10 packages in 204ms
Prepared 2 pa

'/content/drive/MyDrive/TFM/Experimentos/E4_augmentation_100ep_pat20_yolo11n/weights/best.onnx'

In [ ]:
from google.colab import files
onnx_path = '/content/drive/MyDrive/TFM/Experimentos/E4_augmentation_100ep_pat20_yolo11n/weights/best.onnx'
files.download(onnx_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

para cuantizar el .onnx a int8 y volver a probarlo cuantizado en la raspberry para ver si mejoran las métricas

In [ ]:
# 1. Instalar dependencias necesarias tras reiniciar Colab
!pip install --quiet ultralytics onnx onnxruntime

import os
from google.colab import drive, files
from ultralytics import YOLO
import onnxruntime
from onnxruntime.quantization import quantize_dynamic, QuantType

# 2. Conectar con Google Drive
drive.mount('/content/drive')

# 3. Definir rutas (ajusta la carpeta en Drive si fuera necesario)
DIR_ORIGEN = '/content/drive/MyDrive/TFM/Experimentos/E4_augmentation_100ep_pat20_yolo11n/weights'
PT_PATH = os.path.join(DIR_ORIGEN, 'best.pt')
ONNX_PATH = os.path.join(DIR_ORIGEN, 'best.onnx')
INT8_ONNX_PATH = os.path.join(DIR_ORIGEN, 'best_int8.onnx')

# 4. Asegurar la exportación estricta a 640x640 si se parte de best.pt
# Si quieres forzar la regeneración en 640x640, pon regenerar_onnx = True
regenerar_onnx = False

if not os.path.exists(ONNX_PATH) or regenerar_onnx:
    print("Exportando best.pt a ONNX FP32 con resolución 640x640...")
    if not os.path.exists(PT_PATH):
        raise FileNotFoundError(f"No se encontró el archivo: {PT_PATH}")
    model = YOLO(PT_PATH)
    # imgsz=(640, 640) fija explícitamente la dimensión espacial cuadrada
    model.export(format='onnx', imgsz=(640, 640), dynamic=False)

# 5. Aplicar cuantización dinámica a INT8 manteniendo la geometría
print(f"Cuantizando a INT8 el modelo (640x640): {ONNX_PATH} ...")
quantize_dynamic(
    model_input=ONNX_PATH,
    model_output=INT8_ONNX_PATH,
    weight_type=QuantType.QUInt8
)


size_fp32 = os.path.getsize(ONNX_PATH) / (1024 * 1024)
size_int8 = os.path.getsize(INT8_ONNX_PATH) / (1024 * 1024)

print("\n" + "="*45)
print("      CUANTIZACIÓN INT8 (640x640) COMPLETADA")
print("="*45)
print(f"Modelo FP32 original:  {size_fp32:.2f} MB")
print(f"Modelo INT8 optimizado: {size_int8:.2f} MB")
print(f"Reducción de tamaño:   {((size_fp32 - size_int8) / size_fp32) * 100:.1f} %")
print("="*45 + "\n")


print("Iniciando descarga de best_int8.onnx a tu PC...")
files.download(INT8_ONNX_PATH)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Mounted at /content/drive


Cuantizando a INT8 el modelo (640x640): /content/drive/MyDrive/TFM/Experimentos/E4_augmentation_100ep_pat20_yolo11n/weights/best.onnx ...

      CUANTIZACIÓN INT8 (640x640) COMPLETADA
Modelo FP32 original:  10.11 MB
Modelo INT8 optimizado: 2.87 MB
Reducción de tamaño:   71.6 %

Iniciando descarga de best_int8.onnx a tu PC...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>